# Advanced Spark Analytics - Complex Data Processing

This notebook demonstrates advanced Spark analytics capabilities including:
- Complex joins and aggregations
- Window functions and analytics
- Data cleaning and transformation
- Time series analysis
- Advanced SQL operations

## Setup and Data Loading

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Create Spark Session with optimized settings
spark = SparkSession.builder \
    .appName("Advanced Analytics Demo") \
    .master("spark://spark-master:7077") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.adaptive.skewJoin.enabled", "true") \
    .getOrCreate()

print(f"Spark Version: {spark.version}")

In [ ]:
# Load all datasets
sales_df = spark.read.csv("/home/jovyan/data/csv/sales_data.csv", header=True, inferSchema=True)
customers_df = spark.read.csv("/home/jovyan/data/csv/customers.csv", header=True, inferSchema=True)
products_df = spark.read.csv("/home/jovyan/data/csv/products.csv", header=True, inferSchema=True)

# Create temporary views
sales_df.createOrReplaceTempView("sales")
customers_df.createOrReplaceTempView("customers")
products_df.createOrReplaceTempView("products")

print(f"Sales records: {sales_df.count()}")
print(f"Customer records: {customers_df.count()}")
print(f"Product records: {products_df.count()}")

## 1. Data Quality Assessment

In [ ]:
# Check for data quality issues
def data_quality_report(df, name):
    print(f"\n=== Data Quality Report for {name} ===")
    print(f"Total rows: {df.count()}")
    
    # Check for nulls in each column
    for col_name in df.columns:
        null_count = df.filter(col(col_name).isNull()).count()
        if null_count > 0:
            print(f"  {col_name}: {null_count} nulls")
    
    # Check for duplicates
    duplicate_count = df.count() - df.distinct().count()
    if duplicate_count > 0:
        print(f"  Duplicate rows: {duplicate_count}")
    
    print("  ✓ Data quality check completed")

data_quality_report(sales_df, "Sales")
data_quality_report(customers_df, "Customers")
data_quality_report(products_df, "Products")

## 2. Complex Joins and Customer Analytics

In [ ]:
# Create comprehensive customer analytics with joins
customer_analytics = spark.sql("""
    WITH customer_metrics AS (
        SELECT 
            c.customer_id,
            c.first_name,
            c.last_name,
            c.age,
            c.city,
            c.loyalty_tier,
            c.signup_date,
            COUNT(s.transaction_id) as total_orders,
            ROUND(SUM(s.total_amount), 2) as lifetime_value,
            ROUND(AVG(s.total_amount), 2) as avg_order_value,
            MAX(s.date) as last_purchase_date,
            MIN(s.date) as first_purchase_date,
            COUNT(DISTINCT s.category) as categories_purchased
        FROM customers c
        LEFT JOIN sales s ON c.customer_id = s.customer_id
        GROUP BY c.customer_id, c.first_name, c.last_name, c.age, c.city, c.loyalty_tier, c.signup_date
    )
    SELECT *,
           CASE 
               WHEN lifetime_value >= 5000 THEN 'High Value'
               WHEN lifetime_value >= 2000 THEN 'Medium Value'
               WHEN lifetime_value > 0 THEN 'Low Value'
               ELSE 'No Purchases'
           END as value_segment,
           DATEDIFF(CURRENT_DATE(), last_purchase_date) as days_since_last_purchase
    FROM customer_metrics
    ORDER BY lifetime_value DESC
""")

customer_analytics.show(10)
customer_analytics.createOrReplaceTempView("customer_analytics")

## 3. Window Functions for Advanced Analytics

In [ ]:
# Customer ranking and percentiles using window functions
window_spec = Window.orderBy(desc("lifetime_value"))
percentile_window = Window.orderBy("lifetime_value")

customer_rankings = customer_analytics \
    .withColumn("customer_rank", row_number().over(window_spec)) \
    .withColumn("lifetime_value_percentile", 
                percent_rank().over(percentile_window)) \
    .withColumn("running_total_ltv", 
                sum("lifetime_value").over(window_spec.rowsBetween(Window.unboundedPreceding, Window.currentRow)))

customer_rankings.select(
    "customer_id", "first_name", "last_name", 
    "lifetime_value", "customer_rank", 
    "lifetime_value_percentile", "running_total_ltv"
).show(15)

In [ ]:
# Monthly sales trends with window functions
monthly_trends = spark.sql("""
    WITH monthly_sales AS (
        SELECT 
            DATE_FORMAT(date, 'yyyy-MM') as month,
            SUM(total_amount) as monthly_revenue,
            COUNT(*) as monthly_transactions,
            AVG(total_amount) as avg_transaction_value
        FROM sales
        GROUP BY DATE_FORMAT(date, 'yyyy-MM')
        ORDER BY month
    )
    SELECT *,
           LAG(monthly_revenue, 1) OVER (ORDER BY month) as prev_month_revenue,
           ROUND(
               (monthly_revenue - LAG(monthly_revenue, 1) OVER (ORDER BY month)) / 
               LAG(monthly_revenue, 1) OVER (ORDER BY month) * 100, 2
           ) as revenue_growth_pct,
           AVG(monthly_revenue) OVER (
               ORDER BY month 
               ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
           ) as three_month_avg_revenue
    FROM monthly_sales
""")

monthly_trends.show(20)

## 4. Advanced Aggregations and Pivoting

In [ ]:
# Create a pivot table for category performance by region
category_region_pivot = sales_df.groupBy("region") \
    .pivot("category") \
    .agg(round(sum("total_amount"), 2).alias("revenue")) \
    .fillna(0)

category_region_pivot.show()

# Add totals
category_region_with_totals = category_region_pivot \
    .withColumn("total_revenue", 
                expr("Books + Clothing + Electronics + `Home & Garden` + Sports + Toys"))

category_region_with_totals.show()

In [ ]:
# Advanced aggregations with multiple metrics
comprehensive_analysis = spark.sql("""
    SELECT 
        category,
        region,
        COUNT(*) as transaction_count,
        ROUND(SUM(total_amount), 2) as total_revenue,
        ROUND(AVG(total_amount), 2) as avg_transaction,
        ROUND(MIN(total_amount), 2) as min_transaction,
        ROUND(MAX(total_amount), 2) as max_transaction,
        ROUND(STDDEV(total_amount), 2) as stddev_transaction,
        ROUND(PERCENTILE_APPROX(total_amount, 0.5), 2) as median_transaction,
        ROUND(PERCENTILE_APPROX(total_amount, 0.25), 2) as q1_transaction,
        ROUND(PERCENTILE_APPROX(total_amount, 0.75), 2) as q3_transaction
    FROM sales
    GROUP BY category, region
    ORDER BY total_revenue DESC
""")

comprehensive_analysis.show(20)

## 5. Customer Segmentation Analysis

In [ ]:
# RFM Analysis (Recency, Frequency, Monetary)
rfm_analysis = spark.sql("""
    WITH customer_rfm AS (
        SELECT 
            customer_id,
            DATEDIFF(CURRENT_DATE(), MAX(date)) as recency_days,
            COUNT(*) as frequency,
            ROUND(SUM(total_amount), 2) as monetary
        FROM sales
        GROUP BY customer_id
    ),
    rfm_scores AS (
        SELECT *,
            NTILE(5) OVER (ORDER BY recency_days DESC) as recency_score,
            NTILE(5) OVER (ORDER BY frequency ASC) as frequency_score,
            NTILE(5) OVER (ORDER BY monetary ASC) as monetary_score
        FROM customer_rfm
    )
    SELECT *,
        CONCAT(recency_score, frequency_score, monetary_score) as rfm_segment,
        CASE 
            WHEN recency_score >= 4 AND frequency_score >= 4 AND monetary_score >= 4 THEN 'Champions'
            WHEN recency_score >= 3 AND frequency_score >= 3 AND monetary_score >= 3 THEN 'Loyal Customers'
            WHEN recency_score >= 3 AND frequency_score <= 2 THEN 'Potential Loyalists'
            WHEN recency_score <= 2 AND frequency_score >= 3 THEN 'At Risk'
            WHEN recency_score <= 2 AND frequency_score <= 2 THEN 'Lost Customers'
            ELSE 'Others'
        END as customer_segment
    FROM rfm_scores
    ORDER BY monetary DESC
""")

rfm_analysis.show(20)
rfm_analysis.createOrReplaceTempView("rfm_analysis")

In [ ]:
# Customer segment summary
segment_summary = spark.sql("""
    SELECT 
        customer_segment,
        COUNT(*) as customer_count,
        ROUND(AVG(recency_days), 1) as avg_recency,
        ROUND(AVG(frequency), 1) as avg_frequency,
        ROUND(AVG(monetary), 2) as avg_monetary,
        ROUND(SUM(monetary), 2) as total_value
    FROM rfm_analysis
    GROUP BY customer_segment
    ORDER BY total_value DESC
""")

segment_summary.show()

## 6. Time Series Analysis

In [ ]:
# Daily sales patterns with day of week analysis
daily_patterns = spark.sql("""
    SELECT 
        date,
        DAYOFWEEK(date) as day_of_week,
        CASE DAYOFWEEK(date)
            WHEN 1 THEN 'Sunday'
            WHEN 2 THEN 'Monday'
            WHEN 3 THEN 'Tuesday'
            WHEN 4 THEN 'Wednesday'
            WHEN 5 THEN 'Thursday'
            WHEN 6 THEN 'Friday'
            WHEN 7 THEN 'Saturday'
        END as day_name,
        COUNT(*) as daily_transactions,
        ROUND(SUM(total_amount), 2) as daily_revenue,
        ROUND(AVG(total_amount), 2) as avg_transaction_value
    FROM sales
    GROUP BY date, DAYOFWEEK(date)
    ORDER BY date
""")

daily_patterns.show(10)

# Weekly patterns summary
weekly_summary = spark.sql("""
    SELECT 
        day_name,
        AVG(daily_transactions) as avg_daily_transactions,
        AVG(daily_revenue) as avg_daily_revenue,
        AVG(avg_transaction_value) as avg_transaction_value
    FROM (
        SELECT 
            CASE DAYOFWEEK(date)
                WHEN 1 THEN 'Sunday'
                WHEN 2 THEN 'Monday'
                WHEN 3 THEN 'Tuesday'
                WHEN 4 THEN 'Wednesday'
                WHEN 5 THEN 'Thursday'
                WHEN 6 THEN 'Friday'
                WHEN 7 THEN 'Saturday'
            END as day_name,
            COUNT(*) as daily_transactions,
            SUM(total_amount) as daily_revenue,
            AVG(total_amount) as avg_transaction_value
        FROM sales
        GROUP BY date, DAYOFWEEK(date)
    ) daily_stats
    GROUP BY day_name
    ORDER BY avg_daily_revenue DESC
""")

weekly_summary.show()

## 7. Product Performance Analysis

In [ ]:
# Product profitability analysis with cost data
product_profitability = spark.sql("""
    SELECT 
        p.category,
        p.product_name,
        p.cost_price,
        p.suggested_price,
        p.profit_margin,
        COUNT(s.transaction_id) as units_sold,
        ROUND(SUM(s.total_amount), 2) as total_revenue,
        ROUND(SUM(s.quantity * p.cost_price), 2) as total_cost,
        ROUND(SUM(s.total_amount) - SUM(s.quantity * p.cost_price), 2) as total_profit,
        ROUND(
            (SUM(s.total_amount) - SUM(s.quantity * p.cost_price)) / SUM(s.total_amount) * 100, 2
        ) as actual_profit_margin_pct
    FROM products p
    JOIN sales s ON p.product_name = s.product_name AND p.category = s.category
    GROUP BY p.category, p.product_name, p.cost_price, p.suggested_price, p.profit_margin
    ORDER BY total_profit DESC
""")

product_profitability.show()

## 8. Advanced Filtering and Complex Conditions

In [ ]:
# Find customers with complex purchasing patterns
complex_customers = spark.sql("""
    WITH customer_patterns AS (
        SELECT 
            customer_id,
            COUNT(DISTINCT category) as categories_count,
            COUNT(DISTINCT region) as regions_count,
            COUNT(*) as total_purchases,
            SUM(total_amount) as total_spent,
            MAX(total_amount) as max_purchase,
            MIN(total_amount) as min_purchase,
            STDDEV(total_amount) as purchase_variance
        FROM sales
        GROUP BY customer_id
    )
    SELECT 
        cp.*,
        c.first_name,
        c.last_name,
        c.loyalty_tier,
        CASE 
            WHEN categories_count >= 4 AND total_purchases >= 10 THEN 'Diverse Buyer'
            WHEN total_spent >= 3000 AND purchase_variance > 200 THEN 'High Value Variable'
            WHEN regions_count > 1 THEN 'Multi-Region Buyer'
            ELSE 'Regular Customer'
        END as customer_pattern
    FROM customer_patterns cp
    JOIN customers c ON cp.customer_id = c.customer_id
    WHERE categories_count >= 3 OR total_spent >= 2000 OR regions_count > 1
    ORDER BY total_spent DESC
""")

complex_customers.show(20)

## 9. Data Export and Visualization Prep

In [ ]:
# Prepare data for visualization
viz_data = monthly_trends.toPandas()

# Simple visualization of monthly trends
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.plot(viz_data['month'], viz_data['monthly_revenue'])
plt.title('Monthly Revenue Trend')
plt.xticks(rotation=45)
plt.ylabel('Revenue ($)')

plt.subplot(1, 2, 2)
plt.plot(viz_data['month'], viz_data['revenue_growth_pct'])
plt.title('Monthly Revenue Growth %')
plt.xticks(rotation=45)
plt.ylabel('Growth %')
plt.axhline(y=0, color='r', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
# Save analysis results
# Customer analytics
customer_analytics.write.mode("overwrite").parquet("/home/jovyan/data/parquet/customer_analytics")

# RFM analysis
rfm_analysis.write.mode("overwrite").parquet("/home/jovyan/data/parquet/rfm_analysis")

# Product profitability
product_profitability.write.mode("overwrite").parquet("/home/jovyan/data/parquet/product_profitability")

# Monthly trends
monthly_trends.write.mode("overwrite").parquet("/home/jovyan/data/parquet/monthly_trends")

print("All analysis results saved to Parquet format!")

## Summary

This notebook demonstrated advanced Spark analytics including:

1. **Data Quality Assessment**: Checking for nulls, duplicates, and data integrity
2. **Complex Joins**: Multi-table analysis with customer, sales, and product data
3. **Window Functions**: Rankings, running totals, and time-based analytics
4. **Advanced Aggregations**: Pivot tables and comprehensive statistical metrics
5. **Customer Segmentation**: RFM analysis for customer behavior understanding
6. **Time Series Analysis**: Temporal patterns and trend analysis
7. **Product Analytics**: Profitability and performance analysis
8. **Complex Filtering**: Advanced query patterns and customer categorization

## Key Takeaways
- Spark SQL provides powerful analytics capabilities
- Window functions enable sophisticated analytical operations
- Proper data modeling enhances analysis efficiency
- Combining multiple datasets reveals deeper business insights